# OTT Viewing Analysis – Sample Data Analyst Project
**Audience:** B.Tech 1st Year (Data Analyst track) | **Tools:** Python, pandas, NumPy, matplotlib, seaborn

## 1. Business problem
An OTT (streaming) platform wants to understand **what drives viewer satisfaction and engagement**, and to spot **subscribers at risk of leaving**, so the product and retention teams can act early.

**Questions we will answer**
1. What does overall engagement and satisfaction look like?
2. Do watch hours and buffering affect satisfaction?
3. Does satisfaction differ by plan, device, genre, city or age group?
4. Which viewers are at risk (satisfaction score below 50), and how recently were viewers active?

> The dataset is **synthetic** (computer-generated) and contains no real viewer data.

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi']=100

## 3. Load and inspect the data

In [ ]:
df = pd.read_csv('../data/raw/ott_viewing_data.csv')
print('Rows, Columns:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Missing values per column:'); print(df.isna().sum()[df.isna().sum()>0])
print('\nDuplicate rows:', df.duplicated().sum())
print('\nCity spellings:', sorted(df['City'].unique()))
df.describe().round(1)

**What we found:** missing values in 4 columns, 5 duplicate rows, messy `City` text (`delhi`, `DELHI`, ` pune`), and impossible `Age` values (a negative age and 212). We clean these next.

## 4. Clean the data

In [ ]:
clean = df.copy()
# 1. Remove duplicate rows
clean = clean.drop_duplicates()
# 2. Standardise text (strip spaces, consistent capitalisation)
clean['City'] = clean['City'].str.strip().str.title()
# 3. Fix impossible values (a viewer cannot be negative or 212 years old)
clean.loc[(clean['Age']<10)|(clean['Age']>90),'Age'] = np.nan
# 4. Fix data types
clean['Last_Active_Date'] = pd.to_datetime(clean['Last_Active_Date'])
# 5. Fill missing values: median for numbers, 'Unknown' for categories
for c in ['Age','Weekly_Watch_Hours']:
    clean[c] = clean[c].fillna(clean[c].median())
for c in ['Primary_Device','Favourite_Genre']:
    clean[c] = clean[c].fillna('Unknown')
print('Rows after cleaning:', len(clean), '| Missing values left:', clean.isna().sum().sum())
print('Cities:', sorted(clean['City'].unique()))

## 5. Create new columns (feature engineering)

In [ ]:
# Days since the viewer was last active (reference date = latest date in the data)
ref_date = clean['Last_Active_Date'].max()
clean['Days_Since_Active'] = (ref_date - clean['Last_Active_Date']).dt.days

def sat_band(s):
    if s>=75: return 'Delighted'
    if s>=60: return 'Satisfied'
    if s>=50: return 'Neutral'
    return 'Unhappy'
clean['Satisfaction_Band'] = clean['Satisfaction_Score'].apply(sat_band)

# Last-active dates span about 2 years, so we use 90-day and 1-year cut-offs
clean['Activity_Status'] = pd.cut(clean['Days_Since_Active'],[-1,90,365,10000],labels=['Recent (<=90d)','Cooling (91-365d)','Dormant (>1yr)'])
clean['Age_Group'] = pd.cut(clean['Age'],[0,24,34,44,100],labels=['<=24','25-34','35-44','45+'])
clean['Buffering_Band'] = pd.cut(clean['Buffering_Incidents'],[-1,2,5,8,100],labels=['0-2','3-5','6-8','9+'])
# A viewer is "At Risk" if satisfaction is below 50 (same idea as 'average below 50' in a marks project)
clean['At_Risk'] = clean['Satisfaction_Score']<50
clean[['Viewer_ID','Satisfaction_Score','Satisfaction_Band','Days_Since_Active','Activity_Status','At_Risk']].head()

## 6. Exploratory data analysis (EDA)
### Q1. Overall engagement and satisfaction

In [ ]:
print(clean[['Weekly_Watch_Hours','Satisfaction_Score','Buffering_Incidents','Months_Subscribed']].describe().round(1))
order=['Delighted','Satisfied','Neutral','Unhappy']
pal={'Delighted':'#1F3864','Satisfied':'#2E75B6','Neutral':'#9DC3E6','Unhappy':'#C00000'}
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.histplot(clean['Satisfaction_Score'],bins=15,kde=True,color='#1F3864',ax=ax[0]);ax[0].set_title('Distribution of Satisfaction Score')
sns.countplot(data=clean,x='Satisfaction_Band',hue='Satisfaction_Band',legend=False,order=order,palette=pal,ax=ax[1]);ax[1].set_title('Viewers by Satisfaction Band')
plt.tight_layout();plt.savefig('../images/01_overall_satisfaction.png');plt.show()

### Q2. Do watch hours and buffering affect satisfaction?

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
sns.scatterplot(data=clean,x='Weekly_Watch_Hours',y='Satisfaction_Score',alpha=.6,color='#2E75B6',ax=ax[0])
sns.regplot(data=clean,x='Weekly_Watch_Hours',y='Satisfaction_Score',scatter=False,color='#C00000',ax=ax[0]);ax[0].set_title('Watch Hours vs Satisfaction')
sns.scatterplot(data=clean,x='Buffering_Incidents',y='Satisfaction_Score',alpha=.6,color='#2E75B6',ax=ax[1])
sns.regplot(data=clean,x='Buffering_Incidents',y='Satisfaction_Score',scatter=False,color='#C00000',ax=ax[1]);ax[1].set_title('Buffering Incidents vs Satisfaction')
plt.tight_layout();plt.savefig('../images/02_watch_buffering.png');plt.show()
num_cols=['Age','Weekly_Watch_Hours','Buffering_Incidents','Months_Subscribed','Satisfaction_Score']
plt.figure(figsize=(7,5))
sns.heatmap(clean[num_cols].corr().round(2),annot=True,cmap='Blues',fmt='.2f');plt.title('Correlation Heatmap')
plt.tight_layout();plt.savefig('../images/03_correlation.png');plt.show()

In [ ]:
print(clean.groupby('Buffering_Band',observed=True)['Satisfaction_Score'].agg(['count','mean']).round(1))

### Q3. Does it differ by plan, device, genre, city or age group?

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(15,8))
for a,col in zip(ax.flat,['Subscription_Plan','Primary_Device','Favourite_Genre','City','Age_Group']):
    sns.barplot(data=clean,x=col,y='Satisfaction_Score',errorbar=None,color='#2E75B6',ax=a)
    a.set_title(f'Satisfaction by {col}');a.set_ylim(0,100);a.tick_params(axis='x',rotation=20)
    for p in a.patches:a.annotate(f'{p.get_height():.1f}',(p.get_x()+p.get_width()/2,p.get_height()),ha='center',va='bottom',fontsize=9)
ax.flat[-1].axis('off')
plt.tight_layout();plt.savefig('../images/04_group_comparison.png');plt.show()

In [ ]:
print(clean.groupby('Subscription_Plan')[['Weekly_Watch_Hours','Buffering_Incidents','Satisfaction_Score']].mean().round(1))

### Q4. Which viewers are at risk?

In [ ]:
risk = clean[clean['At_Risk']].sort_values('Satisfaction_Score')
print(f'{len(risk)} viewers ({len(risk)/len(clean):.1%}) are At Risk')
print(clean['Activity_Status'].value_counts())
print('Average buffering, At Risk vs others:', round(risk['Buffering_Incidents'].mean(),1), 'vs', round(clean[~clean['At_Risk']]['Buffering_Incidents'].mean(),1))
print('Average watch hours, At Risk vs others:', round(risk['Weekly_Watch_Hours'].mean(),1), 'vs', round(clean[~clean['At_Risk']]['Weekly_Watch_Hours'].mean(),1))
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.countplot(data=clean,x='Activity_Status',hue='Activity_Status',legend=False,palette=['#1F3864','#9DC3E6','#C00000'],order=['Recent (<=90d)','Cooling (91-365d)','Dormant (>1yr)'],ax=ax[0]);ax[0].set_title('Viewers by Activity Status')
(clean.groupby('Subscription_Plan')['At_Risk'].mean()*100).sort_values().plot(kind='bar',color='#2E75B6',ax=ax[1]);ax[1].set_title('% At Risk (satisfaction < 50) by Plan');ax[1].set_ylabel('% At Risk');ax[1].tick_params(axis='x',rotation=0)
plt.tight_layout();plt.savefig('../images/05_risk.png');plt.show()
risk[['Viewer_ID','City','Subscription_Plan','Weekly_Watch_Hours','Buffering_Incidents','Days_Since_Active','Satisfaction_Score']].head(10)

## 7. Key insights (generated from the data)

In [ ]:
r_w=clean['Weekly_Watch_Hours'].corr(clean['Satisfaction_Score']);r_b=clean['Buffering_Incidents'].corr(clean['Satisfaction_Score'])
plan=clean.groupby('Subscription_Plan')['Satisfaction_Score'].mean()
print(f"1. Average satisfaction: {clean['Satisfaction_Score'].mean():.1f}; average watch time: {clean['Weekly_Watch_Hours'].mean():.1f} hrs/week.")
print(f"2. Watch hours vs satisfaction r = {r_w:.2f}; buffering vs satisfaction r = {r_b:.2f}.")
print(f"3. Highest-satisfaction plan: {plan.idxmax()} ({plan.max():.1f}); lowest: {plan.idxmin()} ({plan.min():.1f}).")
print(f"4. {len(risk)} viewers ({len(risk)/len(clean):.0%}) are At Risk (satisfaction below 50).")
print(f"5. {(clean['Activity_Status']=='Dormant (>1yr)').mean():.0%} of viewers were last active more than a year before the latest date in the data.")

## 8. Recommendations
- Treat **buffering** as the top quality issue: flag viewers with 6+ incidents for a network/app-quality check.
- Run **re-engagement campaigns** (recommendations, offers) for viewers who have not been active for 90+ days.
- Review the **Mobile and Basic plans**, which have the lowest satisfaction, for upgrade offers or feature fixes.

> **Remember:** correlation does not prove causation. This analysis shows association, not cause.

## 9. Save the cleaned data

In [ ]:
clean.to_csv('../data/processed/ott_viewing_clean.csv',index=False)
print('Saved cleaned file with', len(clean), 'rows')